# Tracker improvement for presentation

Consider the scenario where the target evolves according to the Langevin model, driven by a normal sigma-mean mixture with the mixing distribution being the $\alpha$-stable distribution.

In [39]:
import sys
import os

# Add the local stonesoup directory to sys.path
project_path = r"C:\Users\joesb\Documents\stonesoup"  # Adjust this to your actual path
if project_path not in sys.path:
    sys.path.insert(0, project_path)

# print(sys.path)

import numpy as np
from datetime import datetime, timedelta
np.random.seed(1991)

The state of the target can be represented as 2D Cartesian coordinates, $\left[x, \dot x, y, \dot y\right]^{\top}$, modelling both its position and velocity. A simple truth path is created with a sampling rate of 1 Hz.

In [40]:
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.base_driver import GaussianResidualApproxCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levylinear import LevyLangevin, CombinedLinearLevyTransitionModel

# And the clock starts
start_time = datetime.now().replace(microsecond=0)

In [41]:
seed = 1 # Random seem for reproducibility

# Driving process parameters
mu_W = 0
sigma_W2 = 4
alpha = 1.4
c=10
noise_case=GaussianResidualApproxCase()

# Model parameters
theta=0.15

driver_x = AlphaStableNSMDriver(mu_W=mu_W, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, noise_case=noise_case)
driver_y = driver_x # Same driving process in both dimensions and sharing the same latents (jumps)
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta, mu_W=-0.02)
langevin_y = LevyLangevin(driver=driver_y, damping_coeff=theta)
transition_model = CombinedLinearLevyTransitionModel([langevin_x, langevin_y])


In [ ]:
print(transition_model.model_list[0].matrix(time_interval=timedelta(seconds=1)))

The ground truth is initialised from (0,0).

In [43]:
timesteps = [start_time]

truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])

num_steps = 100
number_particles = 100

for k in range(num_steps):
    timesteps.append(start_time+timedelta(seconds=1*(k+1)))  # add next timestep to list of timesteps
    truth.append(GroundTruthState(
        transition_model.function(truth[k], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k+1]))
    


## Simulate measurements

In [44]:
from stonesoup.types.detection import Detection
from stonesoup.models.measurement.linear import LinearGaussian

In [45]:
measurement_model = LinearGaussian(
    ndim_state=4,  # Number of state dimensions (position and velocity in 2D)
    mapping=(0, 2),  # Mapping measurement vector index to state index
    noise_covar=np.array([[150, 0],  # Covariance matrix for Gaussian PDF
                          [0, 150]])
    )

The measurements can now be generated and plotted accordingly.

In [46]:
measurements = []
for state in truth:
    measurement = measurement_model.function(state, noise=True)
    measurements.append(Detection(measurement,
                                  timestamp=state.timestamp,
                                  measurement_model=measurement_model))

## Marginalised Particle Filtering

In [47]:
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler 
from stonesoup.updater.particle import MarginalisedParticleUpdater

predictor = MarginalisedParticlePredictor(transition_model=transition_model)
resampler = SystematicResampler()
updater = MarginalisedParticleUpdater(measurement_model, resampler)

In [48]:
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import StateVectors

# Sample from the prior Gaussian distribution
states = multivariate_normal.rvs(np.array([0, 1, 0, 1]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
covars = np.stack([np.eye(4) * 100 for i in range(number_particles)], axis=2) # (M, M, N)

# Create prior particle state.
prior = MarginalisedParticleState(
    state_vector=StateVectors(states.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))

In [ ]:
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

track = Track()

for measurement in measurements:
    prediction = predictor.predict(prior, timestamp=measurement.timestamp)
    hypothesis = SingleHypothesis(prediction, measurement)
    post = updater.update(hypothesis, store_resample_data=True)
    track.append(post)
    prior = track[-1]
    print(f"track length ={len(track)} of {len(measurements)}")

# Implement Particle Smoother Class

In [50]:
from stonesoup.smoother.particle import MarginalisedKalmanSmoother, ParticleSmoother, CarterKohnParticleSmoother
particlesmoother=ParticleSmoother(track=track)
conditionalsmoother=MarginalisedKalmanSmoother(track=track)
CarterKohnparticlesmoother=CarterKohnParticleSmoother(track=track)

### Conditional Kalman Smoother

In [51]:
descendant_smoothed= particlesmoother.smooth()
mean_descendant_smoothed=particlesmoother.mean_track()

In [52]:
Kalman_smoothed_particle_track_list=conditionalsmoother.smooth()
Kalman_mean_track=conditionalsmoother.mean_track()

In [ ]:
CarterKohnsmoothed_particle_track_list=CarterKohnparticlesmoother.smooth()
CarterKohn_mean_track=conditionalsmoother.mean_track()

## Testing Smoothing

In [ ]:
tracking_filters = ["unsmoothed", "mean_descendant_smoothed", "mean_kalman_smoothed","mean_CK_smoothed"]

from stonesoup.metricgenerator.ospametric import OSPAMetric

ospa_generators = [OSPAMetric(c=40, p=1,
                              generator_name=f'{tracking_filter} OSPA metrics',
                              tracks_key=f'tracks_{tracking_filter}',
                              truths_key='truths'
                             )
                   for tracking_filter in tracking_filters]

from stonesoup.metricgenerator.tracktotruthmetrics import SIAPMetrics
from stonesoup.measures import Euclidean

siap_generators = [SIAPMetrics(position_measure=Euclidean((0, 2)),
                             velocity_measure=Euclidean((1, 3)),
                             generator_name=f'{tracking_filter} SIAP metrics',
                             tracks_key=f'tracks_{tracking_filter}',
                             truths_key='truths'
                            )
                  for tracking_filter in tracking_filters]


from stonesoup.metricgenerator.uncertaintymetric import SumofCovarianceNormsMetric

uncertainty_generators = [
    SumofCovarianceNormsMetric(generator_name=f'{tracking_filter} OSPA metrics',
                               tracks_key=f'tracks_{tracking_filter}')
    for tracking_filter in tracking_filters]

from stonesoup.dataassociator.tracktotrack import TrackToTruth
from stonesoup.metricgenerator.manager import MultiManager

associator = TrackToTruth(association_threshold=30)

generators = ospa_generators + siap_generators + uncertainty_generators
metric_manager = MultiManager(generators, associator=associator)

metric_manager.add_data({'truths': [truth],
                         'tracks_unsmoothed': [track],
                         'tracks_mean_descendant_smoothed': [mean_descendant_smoothed],
                        'tracks_mean_kalman_smoothed': [Kalman_mean_track],
                        'tracks_mean_CK_smoothed': [CarterKohn_mean_track]
                         })
metrics = metric_manager.generate_metrics()

from stonesoup.plotter import MetricPlotter

fig1 = MetricPlotter()
fig1.plot_metrics(metrics, metric_names=['OSPA distances'])

In [ ]:
# sum up distance error from ground truth over all timestamps
for tracking_filter in tracking_filters:
    total = sum([metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value[i].value
                 for i in range(0, len(metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value))])
    print(f'OSPA total value for {tracking_filter} is {total:.3f}')

### Monte-Carlo Particle Smoother

### Generate/plot particle track figures in 1D: ground truths/ measurements/ particle_tracks/ unsmoothed_track/ smoothed_tracks

In [56]:
from pathlib import Path
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension

In [ ]:
axis_label_list=["x","dx_dt","y","dy_dt"]
particle_plotter_dict = {}

for i,label in enumerate(axis_label_list):
    particle_plotter_dict[label]= Plotterly(autosize=False, width=1200,height=600, dimension=Dimension.ONE, axis_labels=[label])
    particle_plotter_dict[label].plot_ground_truths(truth, [i],mode="lines", line=dict(width=1))
    if label =="x" or label=="y":
        particle_plotter_dict[label].plot_measurements(measurements, [i],marker=dict(symbol="x",size=4))
    particle_plotter_dict[label].plot_tracks(track, [i],uncertainty=True,particle=True,mode="lines", track_label="Filtered Track",line=dict(width=1))
    particle_plotter_dict[label].plot_tracks(descendant_smoothed,[i],mode="lines",opacity=0.4,track_label="Particle Paths",line=dict(width=0.5))
    particle_plotter_dict[label].plot_tracks(Kalman_smoothed_particle_track_list,[i],mode="lines",opacity=0.4,track_label="Kalman-Smoothed Particle Paths",line=dict(width=0.5))
    particle_plotter_dict[label].plot_tracks(Kalman_mean_track, [i],mode="lines",track_label="Kalman-Smoothed Mean",line=dict(width=0.5))
    particle_plotter_dict[label].plot_tracks(CarterKohnsmoothed_particle_track_list, [i],mode="lines",opacity=0.4,track_label="CK-Smoothed Particle Paths",line=dict(width=0.5))
    particle_plotter_dict[label].plot_tracks(CarterKohn_mean_track, [i],mode="lines",track_label="CK-Smoothed Mean",line=dict(width=0.5))
    # particle_plotter_dict[label].plot_tracks(descendant_smoothed_track, [i],mode="lines",track_label="Descendant Smoothed Track",line=dict(width=1))
    # file_path = Path(rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\TrackingSimulationPlots\{num_steps}steps_{number_particles}p\1D_plot_{label}.html")
    # file_path.parent.mkdir(parents=True, exist_ok=True)
    # particle_plotter_dict[label].fig.write_html(str(file_path))
    particle_plotter_dict[label].fig.show()

### 2D Plot 

In [ ]:
animated=True
animated_label_list= ["xy_position", "xy_velocity"]
animated_mappings =[[0,2],[1,3]]
animated_plotter_dict={}
for i,label in enumerate(animated_label_list):
    if label=="xy_velocity":
        axis_labels=["dx/dt","dy/dt"]
    else:
        axis_labels=["x","y"]
    if animated:
        animated_plotter_dict[label]= AnimatedPlotterly(timesteps, tail_length=1, width=1200, height=1200,
                                                    xaxis=dict(title=dict(text=f"<i>{axis_labels[0]}</i>")),
                                                    yaxis=dict(title=dict(text=f"<i>{axis_labels[1]}</i>")))
    else:
        animated_plotter_dict[label]= Plotterly(autosize=True, axis_labels=axis_labels)
    animated_plotter_dict[label].plot_ground_truths(truth, animated_mappings[i],line=dict(width=1))
    if label== "xy_position":
        animated_plotter_dict[label].plot_measurements(measurements, animated_mappings[i],marker=dict(symbol="x",size=4))
    animated_plotter_dict[label].plot_tracks(track, animated_mappings[i], particle=True, uncertainty=True, track_label="Track",line=dict(width=1))
    animated_plotter_dict[label].plot_tracks(particle_track_list, animated_mappings[i],mode="lines",opacity=0.4,track_label="Particle Paths",line=dict(width=0.5))
    animated_plotter_dict[label].plot_tracks(Kalman_smoothed_particle_track_list, animated_mappings[i],mode="lines",opacity=0.4,track_label="Kalman-Smoothed Particle Paths",line=dict(width=0.5))
    animated_plotter_dict[label].plot_tracks(Kalman_mean_track, animated_mappings[i],mode="lines",track_label="Kalman-Smoothed Mean",line=dict(width=0.5))
    animated_plotter_dict[label].plot_tracks(CarterKohnsmoothed_particle_track_list, animated_mappings[i],mode="lines",opacity=0.4,track_label="CK-Smoothed Particle Paths",line=dict(width=0.5))
    animated_plotter_dict[label].plot_tracks(CarterKohn_mean_track, animated_mappings[i],mode="lines",track_label="CK-Smoothed Mean",line=dict(width=0.5))
    animated_plotter_dict[label].plot_tracks(descendant_smoothed_track, animated_mappings[i],track_label="Descendant Smoothed Track",line=dict(width=1))
    file_path = Path(rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\TrackingSimulationPlots\{num_steps}steps_{number_particles}p\animated_2D_plot_{label}.html")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    # animated_plotter_dict[label].fig.write_html(str(file_path))

### Particle Path Smoother Algorithm

1. **Input**: 
   - Run filtering algorithm to obtain filtered track: $\{\mathbf{x}_t^{(i)}, \omega_t^{(i)}\}_{t=0}^T$ 

2. **Find Resampling history**
   - Call `get_particle_paths` smoother to find chain of particles from which current generation were resampled

3. **Initialization of track object**: 
   - `smoothed_track` = Track()

4. **Reassign particle weights**:
   - For $t = 0$ to $T - 1$:
      - For $n=0$ to $n=num\_particles$
     1. **Calculate Descendant Counts**:
        - Call `get_descendant_count`(based on total number of particle paths which include current particle) to compute number of descendants for each particle at $T-1$.
        - Given resampling, weights are uniform so $w_{t|T}$ $\alpha$ $\text{descendants}_t$
     2. **Compute Normalised Smoothing Weights**:
        - For each particle $i$ at $t$:
          $\rho_t^{(i)} \gets \frac{\text{descendants}_t^{(i)}}{D_t}$
     3. **Append to Smoothed Track**:
        - Append marginalised particle state:
          `smoothed_track` $.append(\{\mathbf{x}_t^{(i)}, \rho_t^{(i)}\})$
          
5. **Output**: 
   - Return `smoothed_track`


### Rao-Blackwellized Particle Filter Algorithm 
1. **Input**:
   - Observations $\{y_{1:T}\}$.
   - Number of particles $N$.
   - Initial prior distribution: $p(x_0^N)$ and associated weights $w_0^{(i)} = \frac{1}{N}$.
   - Transition dynamics: $p(x_t^N | x_{t-1}^N)$.
   - Conditional Gaussian dynamics: $p(x_t^G | x_t^N) \sim \mathcal{N}(\mu_t, P_t)$.
   - Observation likelihood: $p(y_t | x_t^G)$.

---

2. **Initialize**:
   - Sample initial particles $x_0^{N, (i)} \sim p(x_0^N)$ for $i = 1, \ldots, N$.
   - Assign equal weights: $w_0^{(i)} = \frac{1}{N}$.

---

3. **Particle Filtering (Forward Pass)**:
   - For $t = 1, \ldots, T$:
     1. **Particle Propagation**:
        - Propagate non-Gaussian states:
          $$ x_t^{N, (i)} \sim p(x_t^N \mid x_{t-1}^{N, (i)}). $$
        - Predict Gaussian states:
          $$ \mu_{t \mid t-1}^{(i)} = F(x_{t-1}^{N, (i)}) \mu_{t-1}^{(i)}, $$
          $$ P_{t \mid t-1}^{(i)} = F(x_{t-1}^{N, (i)}) P_{t-1}^{(i)} F^\top(x_{t-1}^{N, (i)}) + Q(x_{t-1}^{N, (i)}). $$

     2. **Weight Update**:
        - Update weights using the observation likelihood:
          $$ w_t^{(i)} \propto w_{t-1}^{(i)} \cdot p(y_t \mid \mu_{t \mid t-1}^{(i)}, P_{t \mid t-1}^{(i)}). $$

     3. **Resampling**:
        - Normalize weights:
          $$ \tilde{w}_t^{(i)} = \frac{w_t^{(i)}}{\sum_{j=1}^N w_t^{(j)}}. $$
        - Resample particles $\{x_t^{N, (i)}\}$ based on normalized weights $\{\tilde{w}_t^{(i)}\}$.

     4. **Kalman Filter Correction**:
        - Update Gaussian states conditioned on observations:
          $$ \mu_t^{(i)} = \mu_{t \mid t-1}^{(i)} + K_t^{(i)} \big(y_t - H \mu_{t \mid t-1}^{(i)}\big), $$
          $$ P_t^{(i)} = \big(I - K_t^{(i)} H\big) P_{t \mid t-1}^{(i)}, $$
          $$ K_t^{(i)} = P_{t \mid t-1}^{(i)} H^\top \big(H P_{t \mid t-1}^{(i)} H^\top + R\big)^{-1}. $$

---

4. **Output**:
   - Posterior distribution $\{x_t^{N, (i)}, \mu_t^{(i)}, P_t^{(i)}, w_t^{(i)}\}_{t=1}^T$.
   - Gaussian state estimates $\mu_t^{(i)}$ and $P_t^{(i)}$ conditioned on non-linear states $\{x_t^N\}$.


### Conditional Kalman Smoother Algorithm

1. **Input**:
   - Filtered posterior track $\{\mathbf{x}_t^{(i)}, P_t^{(i)}\}_{t=0}^T$
   - Prediction track $\{\mathbf{x}_{t|t-1}^{(i)}, P_{t|t-1}^{(i)}\}_{t=1}^T$
   - Transition matrices $\{F_t^{(i)}, Q_t^{(i)}\}$, augmented as necessary.

2. **Augment Tracks**:
   - For each particle in the track:
     1. **Augment State Vectors**:
        - Append an extra dimension with value `1` to the state vector $\mathbf{x}_t$.
     2. **Augment Covariance Matrices**:
        - Pad state covariance matrices $P_t$ with a row and column of zeros and set the last diagonal element to `1`.
     3. **Augment Transition Matrices**:
        - Construct:
          $$F_t' = \begin{bmatrix} F_t & \mu_t \\ \mathbf{0} & 1 \end{bmatrix}, \quad Q_t' = \text{padded}(Q_t)$$
          - Where $\mu_t$ is the process mean.

3. **Initialize**:
   - Set `smoothed_track` to an empty Track.
   - Add the last posterior state (no smoothing needed for the final state).

4. **Backward Recursion**:
   - For $t = T-1$ to $0$:
     1. **Retrieve Posterior and Prediction States**:
        - Access $posterior\_state = \mathbf{x}_t^{(i)}$
        - Access $prediction\_state = \mathbf{x}_{t+1|t}^{(i)}$ with augmented matrices $F_t'$ and $Q_t'$.
     2. **Compute Smoothing Gain**:
        - Calculate:
          $$G_t = P_t F_t'^T (P_{t+1|t})^{-1}$$
     3. **Update Smoothed State**:
        - Mean:
          $$\mathbf{x}_{t|T} = \mathbf{x}_t + G_t(\mathbf{x}_{t+1|T} - \mathbf{x}_{t+1|t})$$
        - Covariance:
          $$P_{t|T} = P_t + G_t(P_{t+1|T} - P_{t+1|t})G_t^T$$
     4. **Revert Augmentation**:
        - Remove the augmented row and column to obtain the original dimensions.
     5. **Append Smoothed State**:
        - Add $\{\mathbf{x}_{t|T}, P_{t|T}\}$ to `smoothed_track`.

5. **Output**:
   - Return `smoothed_track`, containing states with refined estimates across all timesteps.

## References
[1] Lemke, Tatjana, and Simon J. Godsill, 'Inference for models with asymmetric α -stable noise processes', in Siem Jan Koopman, and Neil Shephard (eds), Unobserved Components and Time Series Econometrics (Oxford, 2015; online edn, Oxford Academic, 21 Jan. 2016)

[2] S. Godsill, M. Riabiz, and I. Kontoyiannis, “The L ́evy state space model,” in 2019 53rd Asilomar Conference on Signals, Systems, and Computers, 2019, pp. 487–494.
